In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyspark.sql.functions as f
import statsmodels.api as sm
from gentropy.common.session import Session


Loading BokehJS ...

/Users/dc16/Gentropy-manuscript/.venv/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning:

In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.



In [2]:
session = Session(extended_spark_conf={"spark.driver.memory": "10g"})
release_path = "../../../data/25.06/output/"
output_path = "../../../data/intermediate_files/"


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/14 16:14:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/14 16:14:07 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


# Loading inputs


In [3]:
geneConcordance = (
    session.spark.read.parquet(f"{output_path}variants_pleiotropy")
    .withColumn(
        "betaSignConcordance", f.when(f.col("betaSignConcordance").isNull(), 1).otherwise(f.col("betaSignConcordance"))
    )
    .select("variantId", "betaSignConcordance", f.explode("prioritisedGenes").alias("geneId"))
    .groupBy("geneId")
    .agg(
        f.min("betaSignConcordance").alias("betaSignConcordanceMin"),
        f.mean("betaSignConcordance").alias("betaSignConcordanceMean"),
        f.max("betaSignConcordance").alias("betaSignConcordanceMax"),
        f.count("*").alias("numVariants"),
    )
)


In [ ]:
genePS = (
    session.spark.read.parquet(f"{output_path}genes_pleiotropy")
    .select("geneId", "approvedSymbol", "uniqueDiseases", "uniqueTherapeuticAreas")
    .withColumn("log2UniqueDiseases", f.log2("uniqueDiseases"))
)


In [16]:
safety = (
    session.spark.read.parquet(f"{output_path}list_of_genes_32_categories")
    .withColumnRenamed("targetId", "geneId")
    .groupBy("geneId")
    .agg(
        f.max(f.when(f.col("source") == "liable_target", 1).otherwise(0)).alias("liableTarget"),
        f.max(f.when(f.col("source") == "trial_safety_concern", 1).otherwise(0)).alias("trialSafetyConcern"),
    )
)


In [17]:
df = genePS.join(geneConcordance, on="geneId", how="left").join(safety, on="geneId", how="left")
df.count()


8285

In [18]:
(df.show())


+---------------+--------------+--------------+----------------------+------------------+----------------------+-----------------------+----------------------+-----------+------------+------------------+
|         geneId|approvedSymbol|uniqueDiseases|uniqueTherapeuticAreas|log2UniqueDiseases|betaSignConcordanceMin|betaSignConcordanceMean|betaSignConcordanceMax|numVariants|liableTarget|trialSafetyConcern|
+---------------+--------------+--------------+----------------------+------------------+----------------------+-----------------------+----------------------+-----------+------------+------------------+
|ENSG00000001084|          GCLC|             2|                     1|               1.0|                   1.0|                    1.0|                   1.0|          2|           0|                 0|
|ENSG00000002016|         RAD52|             2|                     1|               1.0|                   1.0|                    1.0|                   1.0|          2|           0|

In [ ]:
print(
    "Correlation of unique diseases and minimum beta concordance:",
    df.corr("log2UniqueDiseases", "betaSignConcordanceMin"),
)
print(
    "Correlation of unique diseases and mean beta concordance:",
    df.corr("log2UniqueDiseases", "betaSignConcordanceMean"),
)


Correlation of unique diseases and minimum beta concordance: -0.46020842937517414
Correlation of unique diseases and mean beta concordance: -0.16094262835887513


# Target safety events vs pleiotropy logistic regression


In [10]:
# Convert to pandas and prepare data
df_pd = df.toPandas().copy()

# Create discordance measure (1 - concordance)
df_pd["betaSignDiscordance"] = 1 - df_pd["betaSignConcordanceMin"].fillna(1.0)

# log2(uniqueDiseases)
df_pd["uniqueDiseases_log2"] = np.log2(df_pd["uniqueDiseases"])

covariates = [
    "uniqueDiseases_log2",
    "betaSignDiscordance",
]

# Univariate logistic regression
univariate_results = []
for covariate in covariates:
    x_uni = sm.add_constant(df_pd[[covariate]])
    y_uni = df_pd["liableTarget"]
    model_uni = sm.Logit(y_uni, x_uni).fit(disp=False)
    coef = model_uni.params[covariate]
    se = model_uni.bse[covariate]
    pvalue = model_uni.pvalues[covariate]
    ci_lower = model_uni.conf_int().loc[covariate, 0]
    ci_upper = model_uni.conf_int().loc[covariate, 1]
    univariate_results.append(
        {
            "covariate": covariate,
            "coefficient": coef,
            "std_error": se,
            "p_value": pvalue,
            "ci_lower": ci_lower,
            "ci_upper": ci_upper,
        }
    )
results_df = pd.DataFrame(univariate_results)

# Multivariate logistic regression
x_multi = sm.add_constant(df_pd[covariates])
y_multi = df_pd["liableTarget"]
model_multi = sm.Logit(y_multi, x_multi).fit(disp=False)

multi_results = []
for covariate in covariates:
    coef = model_multi.params[covariate]
    se = model_multi.bse[covariate]
    pvalue = model_multi.pvalues[covariate]
    ci_lower = model_multi.conf_int().loc[covariate, 0]
    ci_upper = model_multi.conf_int().loc[covariate, 1]
    multi_results.append(
        {
            "covariate": covariate,
            "coefficient": coef,
            "std_error": se,
            "p_value": pvalue,
            "ci_lower": ci_lower,
            "ci_upper": ci_upper,
        }
    )
multi_df = pd.DataFrame(multi_results)

print(f"Liable targets: {int(df_pd['liableTarget'].sum())} / {len(df_pd)} genes ({df_pd['liableTarget'].mean():.1%})")

# Combine for plotting
results_df["model_type"] = "Univariate"
multi_df["model_type"] = "Joint"
combined_results = pd.concat([results_df, multi_df], ignore_index=True)
print("Known safety events vs maximum beta discordance:")
print(combined_results.iloc[:, [0, 1, 2, 3, 6]])


Liable targets: 121 / 8285 genes (1.5%)
Known safety events vs maximum beta discordance:
             covariate  coefficient  std_error   p_value  model_type
0  uniqueDiseases_log2     0.142290   0.063432  0.024885  Univariate
1  betaSignDiscordance     1.748956   0.519893  0.000768  Univariate
2  uniqueDiseases_log2     0.057018   0.073423  0.437413       Joint
3  betaSignDiscordance     1.489849   0.620822  0.016404       Joint


In [11]:
# Convert to pandas and prepare data
df_pd = df.toPandas().copy()

# Create discordance measure (1 - concordance)
df_pd["betaSignDiscordance"] = 1 - df_pd["betaSignConcordanceMean"].fillna(1.0)

# log2(uniqueDiseases)
df_pd["uniqueDiseases_log2"] = np.log2(df_pd["uniqueDiseases"])

covariates = [
    "uniqueDiseases_log2",
    "betaSignDiscordance",
]

# Univariate logistic regression
univariate_results = []
for covariate in covariates:
    x_uni = sm.add_constant(df_pd[[covariate]])
    y_uni = df_pd["liableTarget"]
    model_uni = sm.Logit(y_uni, x_uni).fit(disp=False)
    coef = model_uni.params[covariate]
    se = model_uni.bse[covariate]
    pvalue = model_uni.pvalues[covariate]
    ci_lower = model_uni.conf_int().loc[covariate, 0]
    ci_upper = model_uni.conf_int().loc[covariate, 1]
    univariate_results.append(
        {
            "covariate": covariate,
            "coefficient": coef,
            "std_error": se,
            "p_value": pvalue,
            "ci_lower": ci_lower,
            "ci_upper": ci_upper,
        }
    )
results_df = pd.DataFrame(univariate_results)

# Multivariate logistic regression
x_multi = sm.add_constant(df_pd[covariates])
y_multi = df_pd["liableTarget"]
model_multi = sm.Logit(y_multi, x_multi).fit(disp=False)

multi_results = []
for covariate in covariates:
    coef = model_multi.params[covariate]
    se = model_multi.bse[covariate]
    pvalue = model_multi.pvalues[covariate]
    ci_lower = model_multi.conf_int().loc[covariate, 0]
    ci_upper = model_multi.conf_int().loc[covariate, 1]
    multi_results.append(
        {
            "covariate": covariate,
            "coefficient": coef,
            "std_error": se,
            "p_value": pvalue,
            "ci_lower": ci_lower,
            "ci_upper": ci_upper,
        }
    )
multi_df = pd.DataFrame(multi_results)

print(f"Liable targets: {int(df_pd['liableTarget'].sum())} / {len(df_pd)} genes ({df_pd['liableTarget'].mean():.1%})")

# Combine for plotting
results_df["model_type"] = "Univariate"
multi_df["model_type"] = "Joint"
combined_results = pd.concat([results_df, multi_df], ignore_index=True)
print("Known safety events vs mean beta discordance:")
print(combined_results.iloc[:, [0, 1, 2, 3, 6]])


Liable targets: 121 / 8285 genes (1.5%)
Known safety events vs mean beta discordance:
             covariate  coefficient  std_error   p_value  model_type
0  uniqueDiseases_log2     0.142290   0.063432  0.024885  Univariate
1  betaSignDiscordance     1.463249   1.860062  0.431477  Univariate
2  uniqueDiseases_log2     0.137921   0.064178  0.031630       Joint
3  betaSignDiscordance     0.882074   2.081366  0.671715       Joint


In [12]:
# Convert to pandas and prepare data
df_pd = df.toPandas().copy()

# Create discordance measure (1 - concordance)
df_pd["betaSignDiscordance"] = 1 - df_pd["betaSignConcordanceMin"].fillna(1.0)

# log2(uniqueDiseases)
df_pd["uniqueDiseases_log2"] = np.log2(df_pd["uniqueDiseases"])

covariates = [
    "uniqueDiseases_log2",
    "betaSignDiscordance",
]

# Univariate logistic regression
univariate_results = []
for covariate in covariates:
    x_uni = sm.add_constant(df_pd[[covariate]])
    y_uni = df_pd["trialSafetyConcern"]
    model_uni = sm.Logit(y_uni, x_uni).fit(disp=False)
    coef = model_uni.params[covariate]
    se = model_uni.bse[covariate]
    pvalue = model_uni.pvalues[covariate]
    ci_lower = model_uni.conf_int().loc[covariate, 0]
    ci_upper = model_uni.conf_int().loc[covariate, 1]
    univariate_results.append(
        {
            "covariate": covariate,
            "coefficient": coef,
            "std_error": se,
            "p_value": pvalue,
            "ci_lower": ci_lower,
            "ci_upper": ci_upper,
        }
    )
results_df = pd.DataFrame(univariate_results)

# Multivariate logistic regression
x_multi = sm.add_constant(df_pd[covariates])
y_multi = df_pd["trialSafetyConcern"]
model_multi = sm.Logit(y_multi, x_multi).fit(disp=False)

multi_results = []
for covariate in covariates:
    coef = model_multi.params[covariate]
    se = model_multi.bse[covariate]
    pvalue = model_multi.pvalues[covariate]
    ci_lower = model_multi.conf_int().loc[covariate, 0]
    ci_upper = model_multi.conf_int().loc[covariate, 1]
    multi_results.append(
        {
            "covariate": covariate,
            "coefficient": coef,
            "std_error": se,
            "p_value": pvalue,
            "ci_lower": ci_lower,
            "ci_upper": ci_upper,
        }
    )
multi_df = pd.DataFrame(multi_results)

print(
    f"Trial safety concerns: {int(df_pd['trialSafetyConcern'].sum())} / {len(df_pd)} genes ({df_pd['trialSafetyConcern'].mean():.1%})"
)

# Combine for plotting
results_df["model_type"] = "Univariate"
multi_df["model_type"] = "Joint"
combined_results = pd.concat([results_df, multi_df], ignore_index=True)
print("Trial safety concerns vs maximum beta discordance:")
print(combined_results.iloc[:, [0, 1, 2, 3, 6]])


Trial safety concerns: 245 / 8285 genes (3.0%)
Trial safety concerns vs maximum beta discordance:
             covariate  coefficient  std_error       p_value  model_type
0  uniqueDiseases_log2     0.290706   0.043217  1.736184e-11  Univariate
1  betaSignDiscordance     1.822825   0.369028  7.831036e-07  Univariate
2  uniqueDiseases_log2     0.252557   0.050578  5.930772e-07       Joint
3  betaSignDiscordance     0.653606   0.444203  1.411799e-01       Joint


In [13]:
# Convert to pandas and prepare data
df_pd = df.toPandas().copy()

# Create discordance measure (1 - concordance)
df_pd["betaSignDiscordance"] = 1 - df_pd["betaSignConcordanceMean"].fillna(1.0)

# log2(uniqueDiseases)
df_pd["uniqueDiseases_log2"] = np.log2(df_pd["uniqueDiseases"])

covariates = [
    "uniqueDiseases_log2",
    "betaSignDiscordance",
]

# Univariate logistic regression
univariate_results = []
for covariate in covariates:
    x_uni = sm.add_constant(df_pd[[covariate]])
    y_uni = df_pd["trialSafetyConcern"]
    model_uni = sm.Logit(y_uni, x_uni).fit(disp=False)
    coef = model_uni.params[covariate]
    se = model_uni.bse[covariate]
    pvalue = model_uni.pvalues[covariate]
    ci_lower = model_uni.conf_int().loc[covariate, 0]
    ci_upper = model_uni.conf_int().loc[covariate, 1]
    univariate_results.append(
        {
            "covariate": covariate,
            "coefficient": coef,
            "std_error": se,
            "p_value": pvalue,
            "ci_lower": ci_lower,
            "ci_upper": ci_upper,
        }
    )
results_df = pd.DataFrame(univariate_results)

# Multivariate logistic regression
x_multi = sm.add_constant(df_pd[covariates])
y_multi = df_pd["trialSafetyConcern"]
model_multi = sm.Logit(y_multi, x_multi).fit(disp=False)

multi_results = []
for covariate in covariates:
    coef = model_multi.params[covariate]
    se = model_multi.bse[covariate]
    pvalue = model_multi.pvalues[covariate]
    ci_lower = model_multi.conf_int().loc[covariate, 0]
    ci_upper = model_multi.conf_int().loc[covariate, 1]
    multi_results.append(
        {
            "covariate": covariate,
            "coefficient": coef,
            "std_error": se,
            "p_value": pvalue,
            "ci_lower": ci_lower,
            "ci_upper": ci_upper,
        }
    )
multi_df = pd.DataFrame(multi_results)

print(
    f"Trial safety concerns: {int(df_pd['trialSafetyConcern'].sum())} / {len(df_pd)} genes ({df_pd['trialSafetyConcern'].mean():.1%})"
)

# Combine for plotting
results_df["model_type"] = "Univariate"
multi_df["model_type"] = "Joint"
combined_results = pd.concat([results_df, multi_df], ignore_index=True)
print("Trial safety concerns vs mean beta discordance:")
print(combined_results.iloc[:, [0, 1, 2, 3, 6]])


Trial safety concerns: 245 / 8285 genes (3.0%)
Trial safety concerns vs mean beta discordance:
             covariate  coefficient  std_error       p_value  model_type
0  uniqueDiseases_log2     0.290706   0.043217  1.736184e-11  Univariate
1  betaSignDiscordance     1.755643   1.274719  1.684264e-01  Univariate
2  uniqueDiseases_log2     0.288812   0.044034  5.420508e-11       Joint
3  betaSignDiscordance     0.356134   1.611516  8.250979e-01       Joint
